# Customer Behavior Analysis

Цель: изучить поведение покупателей и подготовить данные для анализа повторных покупок.

На первом этапе:
- загружаем оба листа Online Retail II;
- проверяем структуру и качество данных;
- определяем, какие строки относятся к покупкам клиентов.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "online_retail_II.xlsx"

sheets = pd.read_excel(
    data_path,
    sheet_name=["Year 2009-2010", "Year 2010-2011"]
)
df = pd.concat(
    [
        table.assign(SourceSheet=name)
        for name, table in sheets.items()
    ],
    ignore_index=True
)

print(f"Загружено строк: {len(df):,}")

Загружено строк: 1,067,371


In [2]:
source_summary = (
    df.groupby("SourceSheet")
    .agg(
        rows=("Invoice", "size"),
        date_min=("InvoiceDate", "min"),
        date_max=("InvoiceDate", "max"),
        missing_customer_pct=(
            "Customer ID",
            lambda x: x.isna().mean() * 100
        )
    )
)

source_summary["missing_customer_pct"] = source_summary["missing_customer_pct"].round(2)
source_summary

,rows,date_min,date_max,missing_customer_pct
SourceSheet,,,,
Year 2009-2010,525461,2009-12-01 07:45:00,2010-12-09 20:01:00,20.54
Year 2010-2011,541910,2010-12-01 08:26:00,2011-12-09 12:50:00,24.93


## Проверка пересечения двух листов

Оба листа содержат данные за начало декабря 2010 года. Проверим, встречаются
ли одни и те же номера заказов в обоих листах, прежде чем считать число
покупок и суммы по клиентам.

In [3]:
# Номера заказов в каждом исходном листе
invoices_2009 = set(
    df.loc[df["SourceSheet"] == "Year 2009-2010", "Invoice"]
      .astype("string")
)

invoices_2010 = set(
    df.loc[df["SourceSheet"] == "Year 2010-2011", "Invoice"]
      .astype("string")
)

common_invoices = invoices_2009 & invoices_2010

print(f"Номеров заказов, встречающихся в обоих листах: {len(common_invoices):,}")

# Посмотрим на несколько таких заказов и их даты
examples = df.loc[
    df["Invoice"].astype("string").isin(common_invoices),
    ["SourceSheet", "Invoice", "InvoiceDate", "Customer ID", "StockCode", "Quantity", "Price"]
]

display(
    examples
    .sort_values(["Invoice", "SourceSheet", "InvoiceDate"])
    .head(10)
)

Номеров заказов, встречающихся в обоих листах: 1,088


,SourceSheet,Invoice,InvoiceDate,Customer ID,StockCode,Quantity,Price
502938,Year 2009-2010,536365,2010-12-01 08:26:00,17850.0,85123A,6,2.55
502939,Year 2009-2010,536365,2010-12-01 08:26:00,17850.0,71053,6,3.39
502940,Year 2009-2010,536365,2010-12-01 08:26:00,17850.0,84406B,8,2.75
502941,Year 2009-2010,536365,2010-12-01 08:26:00,17850.0,84029G,6,3.39
502942,Year 2009-2010,536365,2010-12-01 08:26:00,17850.0,84029E,6,3.39
502943,Year 2009-2010,536365,2010-12-01 08:26:00,17850.0,22752,2,7.65
502944,Year 2009-2010,536365,2010-12-01 08:26:00,17850.0,21730,6,4.25
525461,Year 2010-2011,536365,2010-12-01 08:26:00,17850.0,85123A,6,2.55
525462,Year 2010-2011,536365,2010-12-01 08:26:00,17850.0,71053,6,3.39
525463,Year 2010-2011,536365,2010-12-01 08:26:00,17850.0,84406B,8,2.75


In [4]:
# Сравниваем строки общих заказов во всех столбцах кроме названия исходного листа
columns_to_compare = df.columns.drop("SourceSheet").tolist()

shared_rows = df.loc[
    df["Invoice"].astype("string").isin(common_invoices)
]

row_counts = (
    shared_rows
    .groupby(columns_to_compare + ["SourceSheet"], dropna=False)
    .size()
    .unstack("SourceSheet", fill_value=0)
)

different_rows = row_counts.loc[
    row_counts["Year 2009-2010"] != row_counts["Year 2010-2011"]
]

print("Строк с различиями между листами:", len(different_rows))
display(different_rows.head(10))

Строк с различиями между листами: 0


,,,,,,,SourceSheet,Year 2009-2010,Year 2010-2011
Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,,


In [5]:
duplicate_mask = (
    df["SourceSheet"].eq("Year 2009-2010")
    & df["Invoice"].astype("string").isin(common_invoices)
)

df_unique = df.loc[~duplicate_mask].copy()

print(f"Удалено повторных строк: {duplicate_mask.sum():,}")
print(f"Строк после объединения без повторов: {len(df_unique):,}")

Удалено повторных строк: 22,523
Строк после объединения без повторов: 1,044,848


## Подготовка покупок клиентов

Оставляем строки с известным покупателем, положительным количеством и ценой.
Отменённые заказы исключаем. Пока сохраняем специальные StockCode для отдельной
проверки: среди них есть доставка, а необычный код не всегда означает ошибку.

### Какие строки считаем покупками клиентов

Нам нужна история операций с положительными количеством и ценой, привязанных к конкретному клиенту.
Поэтому для этой части анализа:

- Исключаем строки без `Customer ID`: мы не можем установить, какому клиенту
  принадлежит операция, и связать её с его следующими заказами.
- Исключаем заказы, у которых `Invoice` начинается с `C`, это отмены, а не
  новые покупки.
- Исключаем строки с `Quantity <= 0`: отрицательное количество отражает
  возврат или корректировку. В первом EDA также были найдены складские
  корректировки без номера отмены.
- Исключаем строки с `Price <= 0`: здесь мы изучаем оплаченные покупки и
  рассчитываем их стоимость. Нулевая цена может встречаться у подарков или
  служебных записей, поэтому мы не называем все такие строки ошибками.

Эти правила задают рабочее определение покупки для Customer Behavior Analysis.
Данные без `Customer ID` ещё могут пригодиться для прогнозирования общих продаж.
Специальные `StockCode`, например `POST` (доставка), проверим отдельно.

In [6]:
cancelled = (
    df_unique["Invoice"]
    .astype("string")
    .str.upper()
    .str.startswith("C", na=False)
)

purchase_mask = (
    df_unique["Customer ID"].notna()
    & ~cancelled
    & (df_unique["Quantity"] > 0)
    & (df_unique["Price"] > 0)
)

purchases = df_unique.loc[purchase_mask].copy()
purchases["LineAmount"] = purchases["Quantity"] * purchases["Price"]

print(f"Строк после удаления повторов: {len(df_unique):,}")
print(f"Строк покупок клиентов: {len(purchases):,}")
print(f"Уникальных клиентов: {purchases['Customer ID'].nunique():,}")
print(f"Уникальных номеров заказов: {purchases['Invoice'].nunique():,}")

Строк после удаления повторов: 1,044,848
Строк покупок клиентов: 791,045
Уникальных клиентов: 5,878
Уникальных номеров заказов: 36,969


## Проверка специальных кодов

В данных о покупках могут оставаться доставка, ручные начисления и другие
операции, которые не являются товарами. Посмотрим на их описание и вклад
в сумму положительных операций, прежде чем рассчитывать стоимость заказов.

In [7]:
standard_code = (
    purchases["StockCode"]
    .astype("string")
    .str.fullmatch(r"\d{5}[A-Za-z]*", na=False)
)

special_summary = (
    purchases.loc[~standard_code]
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        lines=("Invoice", "size"),
        orders=("Invoice", "nunique"),
        amount=("LineAmount", "sum")
    )
    .sort_values("lines", ascending=False)
)

display(special_summary.head(30))

,,lines,orders,amount
StockCode,Description,,,
POST,POSTAGE,1803,1803,124648.040
M,Manual,691,620,152166.320
C2,CARRIAGE,248,248,12546.000
BANK CHARGES,Bank Charges,29,28,420.001
ADJUST,Adjustment by john on 26/01/2010 16,18,18,2107.410
PADS,PADS TO MATCH ALL CUSHIONS,17,17,0.017
DOT,DOTCOM POSTAGE,16,16,11906.360
ADJUST,Adjustment by john on 26/01/2010 17,14,14,1431.110
TEST001,This is a test product.,9,9,225.000


### Отбор товарных покупок

Исключаем из товарных сумм доставку, сборы, корректировки, скидки,
тестовые записи и операции с кодом M (Manual), для которых нельзя
надёжно установить товар. PADS и SP1002 сохраняем: их описания
обозначают товары. Исходную таблицу purchases сохраняем отдельно.

In [8]:
non_product_codes = {
    "POST", "DOT", "C2", "BANK CHARGES",
    "ADJUST", "ADJUST2", "D", "M",
    "TEST001", "TEST002",
}

product_purchases = purchases.loc[
    ~purchases["StockCode"].astype("string").isin(non_product_codes)
].copy()

orders_before = set(purchases["Invoice"].astype("string"))
orders_after = set(product_purchases["Invoice"].astype("string"))
orders_without_products = orders_before - orders_after

print(f"Исключено строк нетоварных операций: {len(purchases) - len(product_purchases):,}")
print(f"Осталось строк товаров: {len(product_purchases):,}")
print(f"Заказов только с нетоварными операциями: {len(orders_without_products):,}")

display(
    purchases.loc[
        purchases["Invoice"].astype("string").isin(orders_without_products),
        ["Invoice", "InvoiceDate", "Customer ID", "StockCode", "Description", "LineAmount"]
    ].head(15)
)

Исключено строк нетоварных операций: 2,839
Осталось строк товаров: 788,206
Заказов только с нетоварными операциями: 375


,Invoice,InvoiceDate,Customer ID,StockCode,Description,LineAmount
126,489444,2009-12-01 09:55:00,12636.0,POST,POSTAGE,141.00
173,489447,2009-12-01 10:10:00,12362.0,POST,POSTAGE,130.00
17386,490760,2009-12-08 09:49:00,14295.0,M,Manual,10.00
17887,490881,2009-12-08 12:58:00,16210.0,M,Manual,10.00
18466,490948,2009-12-08 14:29:00,16805.0,BANK CHARGES,Bank Charges,15.00
19542,490999,2009-12-08 17:26:00,13883.0,M,Manual,15.95
19543,490999,2009-12-08 17:26:00,13883.0,M,Manual,8.70
21450,491094,2009-12-09 13:49:00,12709.0,POST,POSTAGE,216.00
22960,491176,2009-12-10 11:50:00,13091.0,M,Manual,1213.02
27994,491725,2009-12-14 08:34:00,12346.0,TEST001,This is a test product.,45.00


## Таблица товарных заказов

Объединяем товарные строки одного заказа. Сумма заказа здесь равна
сумме положительных товарных строк: доставка и другие нетоварные
операции исключены, а последующие возвраты пока не вычтены.

In [9]:
invoice_check = (
    product_purchases
    .groupby("Invoice")
    .agg(
        customers=("Customer ID", "nunique"),
        dates=("InvoiceDate", "nunique")
    )
)

print("Заказов с несколькими клиентами:",
      (invoice_check["customers"] > 1).sum())
print("Заказов с несколькими датами:",
      (invoice_check["dates"] > 1).sum())

orders = (
    product_purchases
    .groupby(["Invoice", "Customer ID"], as_index=False)
    .agg(
        OrderDate=("InvoiceDate", "min"),
        OrderAmount=("LineAmount", "sum"),
        ProductLines=("StockCode", "size"),
        Units=("Quantity", "sum"),
        DistinctProducts=("StockCode", "nunique")
    )
)

print(f"Товарных заказов: {len(orders):,}")
display(orders.head())

Заказов с несколькими клиентами: 0
Заказов с несколькими датами: 64
Товарных заказов: 36,594


,Invoice,Customer ID,OrderDate,OrderAmount,ProductLines,Units,DistinctProducts
0,489434,13085.0,2009-12-01 07:45:00,505.30,8,166,8
1,489435,13085.0,2009-12-01 07:46:00,145.80,4,60,4
2,489436,13078.0,2009-12-01 09:06:00,630.33,19,193,19
3,489437,15362.0,2009-12-01 09:08:00,310.75,23,145,23
4,489438,18102.0,2009-12-01 09:24:00,2286.24,17,826,17


In [10]:
different_date_invoices = invoice_check.index[
    invoice_check["dates"] > 1
]

date_review = (
    product_purchases.loc[
        product_purchases["Invoice"].isin(different_date_invoices)
    ]
    .groupby("Invoice")
    .agg(
        first_date=("InvoiceDate", "min"),
        last_date=("InvoiceDate", "max"),
        distinct_dates=("InvoiceDate", "nunique"),
        lines=("InvoiceDate", "size")
    )
)

date_review["time_gap"] = (
    date_review["last_date"] - date_review["first_date"]
)

print("Заказов с датами в разные календарные дни:",
      (date_review["first_date"].dt.date
       != date_review["last_date"].dt.date).sum())

display(date_review.sort_values("time_gap", ascending=False).head(15))

Заказов с датами в разные календарные дни: 0


,first_date,last_date,distinct_dates,lines,time_gap
Invoice,,,,,
521901,2010-09-09 12:35:00,2010-09-09 12:37:00,2,49,0 days 00:02:00
492807,2009-12-20 12:28:00,2009-12-20 12:29:00,2,30,0 days 00:01:00
534948,2010-11-24 15:31:00,2010-11-24 15:32:00,2,15,0 days 00:01:00
540185,2011-01-05 13:40:00,2011-01-05 13:41:00,2,29,0 days 00:01:00
541596,2011-01-19 16:18:00,2011-01-19 16:19:00,2,103,0 days 00:01:00
541631,2011-01-20 10:47:00,2011-01-20 10:48:00,2,71,0 days 00:01:00
541816,2011-01-21 15:56:00,2011-01-21 15:57:00,2,73,0 days 00:01:00
541849,2011-01-23 13:33:00,2011-01-23 13:34:00,2,49,0 days 00:01:00
542217,2011-01-26 12:35:00,2011-01-26 12:36:00,2,18,0 days 00:01:00


## Сводка по покупателям

Для каждого клиента считаем число товарных заказов, даты первой и последней
покупки и сумму положительных товарных строк. Повторным покупателем называем
клиента с двумя или более заказами в наблюдаемом периоде.

In [11]:
customers = (
    orders
    .groupby("Customer ID", as_index=False)
    .agg(
        OrderCount=("Invoice", "size"),
        FirstOrder=("OrderDate", "min"),
        LastOrder=("OrderDate", "max"),
        TotalAmount=("OrderAmount", "sum"),
        AvgOrderAmount=("OrderAmount", "mean")
    )
)

customers["RepeatCustomer"] = customers["OrderCount"] >= 2

print(f"Покупателей с товарными заказами: {len(customers):,}")
print(
    "Доля покупателей с двумя и более заказами: "
    f"{customers['RepeatCustomer'].mean():.1%}"
)

display(customers.head())
display(customers["OrderCount"].describe())

Покупателей с товарными заказами: 5,852
Доля покупателей с двумя и более заказами: 72.4%


,Customer ID,OrderCount,FirstOrder,LastOrder,TotalAmount,AvgOrderAmount,RepeatCustomer
0,12346.0,3,2010-03-02 13:08:00,2011-01-18 10:01:00,77352.96,25784.32000,True
1,12347.0,8,2010-10-31 14:20:00,2011-12-07 15:52:00,4921.53,615.19125,True
2,12348.0,5,2010-09-27 14:59:00,2011-09-25 13:13:00,1658.40,331.68000,True
3,12349.0,3,2010-04-29 13:20:00,2011-11-21 09:51:00,3678.69,1226.23000,True
4,12350.0,1,2011-02-02 16:01:00,2011-02-02 16:01:00,294.40,294.40000,False


count    5852.000000
mean        6.253247
std        12.749286
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       373.000000
Name: OrderCount, dtype: float64

## Время между товарными заказами

Для каждого клиента сортируем заказы по времени и считаем интервал от
предыдущего заказа. Первый заказ клиента не имеет предыдущего, поэтому
интервал для него не определён. Анализируем только наблюдаемые интервалы:
если клиент больше не вернулся до конца датасета, его будущий интервал
нам неизвестен.

In [12]:
orders_timeline = orders.sort_values(
    ["Customer ID", "OrderDate", "Invoice"]
).copy()

orders_timeline["PreviousOrderDate"] = (
    orders_timeline
    .groupby("Customer ID")["OrderDate"]
    .shift(1)
)

orders_timeline["DaysSincePreviousOrder"] = (
    (
        orders_timeline["OrderDate"]
        - orders_timeline["PreviousOrderDate"]
    )
    .dt.total_seconds()
    / (24 * 60 * 60)
)

observed_gaps = orders_timeline["DaysSincePreviousOrder"].dropna()

print(f"Наблюдаемых интервалов между заказами: {len(observed_gaps):,}")
print(f"Медианный интервал: {observed_gaps.median():.1f} дня")
print(f"Интервалов не более 90 дней: {(observed_gaps <= 90).mean():.1%}")

display(
    observed_gaps.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

Наблюдаемых интервалов между заказами: 30,742
Медианный интервал: 25.0 дня
Интервалов не более 90 дней: 82.8%


count    30742.000000
mean        52.121112
std         75.903852
min          0.000000
25%          7.023785
50%         25.033333
75%         62.078646
90%        135.971597
95%        207.033090
max        714.152083
Name: DaysSincePreviousOrder, dtype: float64

## Возврат нового клиента за 90 дней

Считаем, появился ли второй товарный заказ в течение 90 дней после первого.
В расчёт доли включаем только клиентов, для которых доступны полные 90 дней
после первой покупки. Это описательная метрика, а не обучающая выборка модели.

In [13]:
orders_timeline["OrderNumber"] = (
    orders_timeline.groupby("Customer ID").cumcount() + 1
)

second_orders = (
    orders_timeline.loc[
        orders_timeline["OrderNumber"] == 2,
        ["Customer ID", "OrderDate"]
    ]
    .rename(columns={"OrderDate": "SecondOrder"})
)

customer_return = customers[
    ["Customer ID", "FirstOrder"]
].merge(
    second_orders,
    on="Customer ID",
    how="left"
)

last_observed_date = orders["OrderDate"].max()

eligible_90d = customer_return.loc[
    customer_return["FirstOrder"]
    <= last_observed_date - pd.Timedelta(days=90)
].copy()

eligible_90d["ReturnedWithin90Days"] = (
    eligible_90d["SecondOrder"].notna()
    & (
        eligible_90d["SecondOrder"]
        <= eligible_90d["FirstOrder"] + pd.Timedelta(days=90)
    )
)

print(f"Клиентов с полными 90 днями наблюдения: {len(eligible_90d):,}")
print(
    "Вернулись за 90 дней после первой покупки: "
    f"{eligible_90d['ReturnedWithin90Days'].mean():.1%}"
)

Клиентов с полными 90 днями наблюдения: 5,256
Вернулись за 90 дней после первой покупки: 46.6%


## RFM-показатели

- Recency — сколько календарных дней прошло с последнего товарного заказа;
- Frequency — сколько товарных заказов сделал клиент;
- Monetary — сумма положительных товарных строк во всех его заказах.

Для описательной сводки берём дату отсчёта на следующий день после
последней даты датасета. Эти показатели используют всю историю и пока
не являются признаками модели на произвольную дату прогноза.

In [14]:
analysis_date = (
    orders["OrderDate"].max().normalize()
    + pd.Timedelta(days=1)
)

rfm = customers[
    ["Customer ID", "LastOrder", "OrderCount", "TotalAmount"]
].copy()

rfm["RecencyDays"] = (
    analysis_date - rfm["LastOrder"].dt.normalize()
).dt.days

rfm = rfm.rename(
    columns={
        "OrderCount": "Frequency",
        "TotalAmount": "Monetary"
    }
)

display(rfm.head())

display(
    rfm[["RecencyDays", "Frequency", "Monetary"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .round(2)
)

,Customer ID,LastOrder,Frequency,Monetary,RecencyDays
0,12346.0,2011-01-18 10:01:00,3,77352.96,326
1,12347.0,2011-12-07 15:52:00,8,4921.53,3
2,12348.0,2011-09-25 13:13:00,5,1658.40,76
3,12349.0,2011-11-21 09:51:00,3,3678.69,19
4,12350.0,2011-02-02 16:01:00,1,294.40,311


,RecencyDays,Frequency,Monetary
count,5852.00,5852.00,5852.00
mean,200.73,6.25,2926.34
std,208.52,12.75,14312.46
min,1.00,1.00,2.95
25%,26.00,1.00,342.20
50%,96.00,3.00,860.26
75%,380.00,7.00,2247.55
90%,531.00,13.00,5393.08
95%,624.00,20.00,9270.27
max,739.00,373.00,580987.04


## Выводы по поведению покупателей

После отбора товарных операций получили 36 594 заказа от 5 852 клиентов.
У 72,4% клиентов наблюдается не менее двух заказов за весь период данных.
Этот показатель нельзя трактовать как вероятность возврата за конкретный срок.

Медианное время между наблюдаемыми заказами - 25 дней. Половина интервалов
не превышает примерно 25 дней, а 75% - примерно 62 дня. Среди наблюдаемых
интервалов 82,8% составляют не более 90 дней. Этот расчёт не учитывает
неизвестные интервалы клиентов, которые ещё не вернулись.

Для 5 256 клиентов после первого заказа доступны полные 90 дней
наблюдения. Из них 46,6% сделали второй товарный заказ в течение 90 дней.
Таким образом, 90 дней - разумный первоначальный горизонт для задачи
прогнозирования повторной покупки.

На дату после окончания наблюдений медианный Recency составляет 96 дней,
медианная Frequency - 3 заказа, медианная Monetary - 860,26. Распределения
частоты и суммы покупок неравномерны: у небольшой части клиентов показатели
намного выше типичных значений. Например, максимальная частота - 373 заказа,
а максимальная сумма положительных товарных строк - 580 987,04.

Денежные показатели включают только положительные товарные строки:
доставка и выбранные нетоварные операции исключены, последующие возвраты
не вычтены. Для будущей модели RFM-признаки нужно будет пересчитывать
отдельно на каждую дату прогноза, используя только уже известные к ней заказы.